In [1]:
# 1. 创建 Keras 官方存放模型的隐藏文件夹
!mkdir -p /home/ma-user/.keras/models/

# 2. 将当前目录下的文件移动/复制过去
!cp mobilenet_v2_weights_tf_dim_ordering_tf_kernels_1.0_224_no_top.h5 /home/ma-user/.keras/models/

# 3. 检查是否移动成功
!ls /home/ma-user/.keras/models/

mobilenet_v2_weights_tf_dim_ordering_tf_kernels_1.0_224_no_top.h5
vgg19_weights_tf_dim_ordering_tf_kernels_notop.h5


In [2]:
import os
import tensorflow as tf

def fix_h5py_decode_error():
    try:
        original_getitem = h5py._hl.attrs.AttributeManager.__getitem__
        def patched_getitem(self, name):
            val = original_getitem(self, name)
            if isinstance(val, str):
                return val.encode('utf-8')
            return val
        h5py._hl.attrs.AttributeManager.__getitem__ = patched_getitem
        print("✅ 兼容性补丁已就绪")
    except:
        pass

fix_h5py_decode_error()

import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import matplotlib.pyplot as plt

# ------------------------------
# 1. 定义 ECA 注意力模块 (修复 TF 2.1 兼容性)
# ------------------------------
class ECAModule(layers.Layer):
    def __init__(self, **kwargs):
        super(ECAModule, self).__init__(**kwargs)
        self.avg_pool = layers.GlobalAveragePooling2D()
        self.conv = None
        self.kernel_size = None

    def build(self, input_shape):
        channel = input_shape[-1]
        k = int(abs((tf.math.log(float(channel), 2) + 1) / 2))
        self.kernel_size = k if k % 2 else k + 1
        self.conv = layers.Conv1D(1, kernel_size=self.kernel_size, padding="same", use_bias=False)
        super(ECAModule, self).build(input_shape)

    def call(self, inputs):
        y = self.avg_pool(inputs)
        y = tf.expand_dims(y, axis=-1)
        y = self.conv(y)
        y = tf.nn.sigmoid(y)
        y = tf.transpose(y, perm=[0, 2, 1])
        y = tf.expand_dims(y, axis=1)
        return inputs * y

    def get_config(self):
        config = super(ECAModule, self).get_config()
        return config

# ------------------------------
# 2. 构建 MobileNetV2 + ECA 模型
# ------------------------------
def build_mobilenet_eca(num_classes=8):
    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base_model.trainable = False

    x = base_model.output
    x = ECAModule()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs=base_model.input, outputs=outputs)
    return model

# ------------------------------
# 3. 数据准备 (保持您的目录结构)
# ------------------------------
train_dir = 'train_test_no'        # 训练集目录，内部按类别存放
val_dir = 'test_0.2'               # 验证集目录
num_classes = 8
batch_size = 8                     # MobileNet 较轻量，可适当增大
epochs = 150

# 训练数据增强（适合雷达图像）
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.05,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=batch_size,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(224, 224),
    batch_size=batch_size,
    class_mode='categorical'
)

# 输出类别映射
class_indices = train_generator.class_indices
print("类别映射:", class_indices)

# ------------------------------
# 4. 编译与训练
# ------------------------------
# GPU 内存增长设置（避免 TensorFlow 2.1 显存占用问题）
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

model = build_mobilenet_eca(num_classes)
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

# 回调函数：早停、学习率衰减、保存最佳模型
checkpoint_path = 'best_mobilenet_eca.h5'
callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-6),
    ModelCheckpoint(checkpoint_path, monitor='val_accuracy', save_best_only=True, mode='max')
]

history = model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    epochs=epochs,
    validation_data=val_generator,
    validation_steps=len(val_generator),
    callbacks=callbacks,
    verbose=1
)

# ------------------------------
# 5. 可视化训练过程 (保留原代码风格)
# ------------------------------
plt.figure(figsize=(12, 5))
plt.rcParams['font.sans-serif'] = ['SimHei']  # 设置字体支持中文
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='训练准确率')
plt.plot(history.history['val_accuracy'], label='验证准确率')
plt.title('准确率-轮次曲线')
plt.xlabel('训练轮次')
plt.ylabel('准确率')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='训练损失值')
plt.plot(history.history['val_loss'], label='验证损失值')
plt.title('损失值-轮次曲线')
plt.xlabel('训练轮次')
plt.ylabel('损失值')
plt.legend()
plt.tight_layout()
plt.show()

# ------------------------------
# 6. 保存最终模型 (可选)
# ------------------------------
model.save('mobilenet_eca_final.h5')
print("训练完成，模型已保存。")

2026-03-23 02:26:54.205471: I tensorflow/stream_executor/platform/default/dso_loader.cc:44] Successfully opened dynamic library libnvinfer.so.6
2026-03-23 02:26:54.207350: I tensorflow/stream_executor/platform/default/dso_loader.cc:44] Successfully opened dynamic library libnvinfer_plugin.so.6
/home/ma-user/anaconda3/envs/TensorFlow-2.1/lib/python3.7/site-packages/requests/__init__.py:104: RequestsDependencyWarning: urllib3 (1.26.12) or chardet (5.2.0)/charset_normalizer (2.0.12) doesn't match a supported version!
  RequestsDependencyWarning)


Found 292 images belonging to 8 classes.
Found 73 images belonging to 8 classes.
类别映射: {'class_hammer': 0, 'class_knife': 1, 'class_pistol': 2, 'class_plier': 3, 'class_rifle': 4, 'class_snips': 5, 'class_stiletto': 6, 'class_wrench': 7}


2026-03-23 02:26:56.366501: I tensorflow/stream_executor/platform/default/dso_loader.cc:44] Successfully opened dynamic library libcuda.so.1
2026-03-23 02:26:56.372448: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2026-03-23 02:26:56.373108: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1555] Found device 0 with properties: 
pciBusID: 0000:00:0d.0 name: Tesla P100-PCIE-16GB computeCapability: 6.0
coreClock: 1.3285GHz coreCount: 56 deviceMemorySize: 15.90GiB deviceMemoryBandwidth: 681.88GiB/s
2026-03-23 02:26:56.373152: I tensorflow/stream_executor/platform/default/dso_loader.cc:44] Successfully opened dynamic library libcudart.so.10.1
2026-03-23 02:26:56.373196: I tensorflow/stream_executor/platform/default/dso_loader.cc:44] Successfully opened dynamic library libcublas.so.10
2026-03-23 02:26:56.375519: I tensorflow/stream_executor/plat

AttributeError: 'str' object has no attribute 'decode'